# Unified IID Experiment — Score-Based Weak-Target Detection
**Single-class + Multi-class, one consistent protocol.** Additive target model only.

This notebook runs the full IID experiment end-to-end on Colab and extracts **each figure
separately** (for LaTeX assembly). It is **resumable**: re-running after a disconnect skips
already-trained models. All models, raw scores, ROC curves and metrics are saved.

**Pipeline**
1. Setup (Drive, deps, paths, device)
2. Configs — 4 runs: `single_n`, `single_rho`, `multi_n`, `multi_rho`
3. Train the 4 sweeps (AMF · DSM · LRao · GMM-Levin(+oracle))
4. Rescore → save raw scores/ROC/metrics → extract per-figure PDFs
5. Study A — DSM depends on preprocessing/scaling
6. Study B — LRao sensitivity (fast convergence · poor multiclass · degrades with n)
7. Figure gallery (each figure shown separately)
8. Package results to Drive

> The *only* deliberate difference between single- and multi-class is the score network:
> **linear** for the near-unimodal single class, **MLP [64,64]** for the multimodal mixture.
> Everything else (epochs, lr, normalisation, n-grid, rho-grid, seeds, metric) is identical.

## 1. Setup

In [ ]:
# Setup: mount Drive, clone the repo, set paths + device, place dataset.
import os, sys, subprocess

try:
    import google.colab          # noqa
    ON_COLAB = True
except Exception:
    ON_COLAB = False
print('Colab:', ON_COLAB)

# >>> EDIT THESE for your setup >>>
REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
REPO_BRANCH   = 'iid-unified-experiment'
# Drive paths (Colab): results persist here; the dataset is copied from here.
DRIVE_RESULTS = '/content/drive/MyDrive/final_paper_experiment/iid_results'
DRIVE_DATASET = '/content/drive/MyDrive/final_paper_experiment/pavia-u.mat'
# <<<

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/proj'
    if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
                        REPO_URL, PROJECT_DIR], check=True)
    else:
        subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=False)
    RESULTS_DIR = DRIVE_RESULTS
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'pymupdf', 'scikit-learn', 'pyyaml'], check=False)
else:
    PROJECT_DIR = os.path.abspath('.')
    RESULTS_DIR = os.path.join(PROJECT_DIR, 'experiments/honest_pipeline/results')

os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Dataset: the repo ships only a (broken-on-Colab) symlink; copy the real .mat.
DATA = 'data/pavia-u.mat'
os.makedirs('data', exist_ok=True)
if not os.path.exists(DATA):
    if os.path.islink(DATA):
        os.remove(DATA)                       # drop broken symlink
    if ON_COLAB and os.path.exists(DRIVE_DATASET):
        import shutil; shutil.copy(DRIVE_DATASET, DATA)
        print('dataset copied from Drive')
assert os.path.exists(DATA), (
    f'pavia-u.mat missing. On Colab put it at {DRIVE_DATASET}; '
    f'locally put it at {PROJECT_DIR}/data/pavia-u.mat')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Project:', PROJECT_DIR)
print('Results:', RESULTS_DIR)
print('Device :', DEVICE, '| torch', torch.__version__)
print('Dataset:', DATA, 'OK')

## 2. Configs
The four configs live in `experiments/honest_pipeline/configs/iid/`. We patch `results_dir` to the (Drive) results path and keep the stable `run_name` so the run is resumable.

In [ ]:
import yaml, glob

CFG_SRC = 'experiments/honest_pipeline/configs/iid'
CFG_RUN = os.path.join(RESULTS_DIR, '_configs')
os.makedirs(CFG_RUN, exist_ok=True)

RUNS = {}   # name -> (config_path, run_dir)
for name in ['single_n', 'single_rho', 'multi_n', 'multi_rho']:
    cfg = yaml.safe_load(open(os.path.join(CFG_SRC, f'{name}.yaml')))
    cfg['results_dir'] = RESULTS_DIR
    cfg['dataset'] = DATA
    out = os.path.join(CFG_RUN, f'{name}.yaml')
    yaml.dump(cfg, open(out, 'w'), sort_keys=False)
    RUNS[name] = (out, os.path.join(RESULTS_DIR, cfg['run_name']))
    print(f"{name:10s}  net={cfg['hidden_dims'] or 'linear':>9}  bkg={cfg['bkg_cls']}  "
          f"n={cfg['n_train_list']}  |rho|={len(cfg['rho_list'])}  gmm={cfg.get('run_gmm_glrt', False)}")
print('\nrun dirs ->', {k: v[1] for k, v in RUNS.items()})

## 3. Train the four sweeps (resumable)
Each call trains AMF / DSM / LRao (and GMM-Levin + oracle for multiclass) over the swept axis, saving every model + pipeline. Re-running skips finished checkpoints.

In [ ]:
import subprocess, sys

def run_sweep(name):
    cfg_path, run_dir = RUNS[name]
    print(f"\n{'='*70}\n  SWEEP: {name}  ->  {run_dir}\n{'='*70}", flush=True)
    subprocess.run([sys.executable, '-u',
                    'experiments/honest_pipeline/run_sweep.py',
                    '--config', cfg_path], check=True)

for name in ['single_n', 'single_rho', 'multi_n', 'multi_rho']:
    run_sweep(name)
print('\nAll sweeps done.')

## 4. Rescore → raw scores + ROC + metrics → per-figure PDFs
`make_pauc_figures.py` reloads every model, reproduces the exact splits, recomputes ROC, dumps raw scores (`<run>/raw_scores/*.npz`) and emits each figure as its own PDF into `pauc_figures/`.

In [ ]:
FIG_DIR = os.path.join(RESULTS_DIR, 'pauc_figures')
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/make_pauc_figures.py',
                '--single_n',   RUNS['single_n'][1],
                '--single_rho', RUNS['single_rho'][1],
                '--multi_n',    RUNS['multi_n'][1],
                '--multi_rho',  RUNS['multi_rho'][1],
                '--out',        FIG_DIR], check=True)
print('Figures + caches ->', FIG_DIR)

In [ ]:
# Combined 2x2 paper figure (additive-only). Reads the caches just produced.
import os
os.environ['PAUC_CACHE_DIR'] = FIG_DIR    # plot_paper_2x2 reads from BASE/pauc_figures
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/plot_paper_2x2.py'], check=False)

## 5. Study A — DSM depends on preprocessing / scaling
Shows DSM fails on raw 103-D and on un-scaled PCA, but recovers with per-band standardisation.

In [ ]:
STUDY_DSM = os.path.join(RESULTS_DIR, 'study_dsm_preproc')
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/study_dsm_preprocessing.py',
                '--out', STUDY_DSM,
                '--seeds', '42', '43', '44', '45', '46'], check=True)

## 6. Study B — LRao sensitivity
(a) converges in a few epochs, (b) trails on multiclass, (c) AUC drops as n grows on multiclass.

In [ ]:
STUDY_LRAO = os.path.join(RESULTS_DIR, 'study_lrao')
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/study_lrao_sensitivity.py',
                '--out', STUDY_LRAO,
                '--single_cache', os.path.join(FIG_DIR, 'cache_single_n.pkl'),
                '--multi_cache',  os.path.join(FIG_DIR, 'cache_multi_n.pkl'),
                '--seeds', '42', '43', '44'], check=True)

## 7. Figure gallery — each figure separately
Every PDF is rendered inline with its filename caption, so it maps 1:1 to a LaTeX `\\includegraphics`.

In [ ]:
import glob, fitz   # PyMuPDF
from IPython.display import Image, display, Markdown

def show_pdf(path, dpi=130):
    doc = fitz.open(path)
    pix = doc[0].get_pixmap(dpi=dpi)
    png = path.replace('.pdf', '.png')
    pix.save(png)
    display(Markdown(f'**`{os.path.relpath(path, RESULTS_DIR)}`**'))
    display(Image(filename=png))

pdfs = []
for sub in ['pauc_figures', 'study_dsm_preproc', 'study_lrao']:
    pdfs += sorted(glob.glob(os.path.join(RESULTS_DIR, sub, '*.pdf')))
pdfs += sorted(glob.glob(os.path.join(RESULTS_DIR, 'paper_2x2.pdf')))
print(f'{len(pdfs)} figures:')
for p in pdfs:
    try:
        show_pdf(p)
    except Exception as e:
        print('  (could not render', p, ':', e, ')')

## 8. Package results to Drive
Zips models + raw scores + caches + metrics + figures for download / archival.

In [ ]:
import shutil
zip_base = os.path.join(RESULTS_DIR, 'iid_experiment_bundle')
# Bundle only the IID run dirs + figures + studies (skip unrelated old runs).
import tempfile, os
stage = tempfile.mkdtemp()
for name in ['single_n', 'single_rho', 'multi_n', 'multi_rho']:
    src = RUNS[name][1]
    if os.path.exists(src):
        shutil.copytree(src, os.path.join(stage, os.path.basename(src)))
for sub in ['pauc_figures', 'study_dsm_preproc', 'study_lrao']:
    s = os.path.join(RESULTS_DIR, sub)
    if os.path.exists(s):
        shutil.copytree(s, os.path.join(stage, sub))
for f in ['paper_2x2.pdf']:
    s = os.path.join(RESULTS_DIR, f)
    if os.path.exists(s): shutil.copy(s, stage)
shutil.make_archive(zip_base, 'zip', stage)
print('Bundle ->', zip_base + '.zip')